<a href="https://colab.research.google.com/github/cyberirishman/5-day-AI-Cyber/blob/main/Lab3a_Cleaning_Homes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Day 2 · Lab 3a — Cleaning: Prove It Changes the Answer

**AI for Cybersecurity Professionals · Day 2: AI for Defense**

In **Lab 3** you caught attacks using hand-written rules. Every rule assumed the data was
*clean*. Real data almost never is. Before any model or rule can be trusted, the data has to
be cleaned — and cleaning is **not** just tidying up. It can flip an answer from nonsense to
sensible.

We use a deliberately simple, non-security dataset — a table of **houses** — so the only
lesson here is *what cleaning does*. (In **Lab 3c** you apply the exact same skills to the
security logs.)

### The four kinds of "dirt" we planted on purpose
1. **Wrong types / units** — prices stored as text like `"$300,000"`, negative ages, `40` bedrooms.
2. **Missing values** — blank prices and blank sizes.
3. **Outliers** — three "extra-zeros" typos (`$50M`, `$62M` and `$88M`), a `$5` house, a `99999` sqft placeholder.
4. **Duplicate rows** — the same house imported several times.

### The anchor question we keep re-asking
> **"What is the mean (average) price of a 4-bedroom house?"**

We will answer it on the dirty data (absurd), then re-answer after **each** cleaning step and
watch the number converge to something believable.

### What you'll do in each step

| Step | What happens |
|---|---|
| **1** | Load the libraries: **pandas** (tables), **numpy** (maths & blanks), **matplotlib** (charts), **scikit-learn** (the model). |
| **2** | Load the dirty house data **straight from GitHub** — nothing to upload. |
| **3** | Look at what *type* each column is, and find the prices that are secretly text. |
| **3b** | Ask the anchor question anyway, using a crude trick — and get an absurd answer. |
| **4** | Fix wrong types — turn `"$300,000"` text into real numbers. |
| **5** | Handle missing values. |
| **6** | Drop physically impossible rows (negative age, 40 bedrooms, 99999 sqft). |
| **7** | Remove outliers with the **IQR** rule. |
| **8** | Remove duplicate rows. |
| **9** | Check whether cleaning changes a **model** too — fit a linear regression, dirty vs. clean. |
| **10** | See it: a dirty-vs-clean chart. |

> **Nothing to download or upload.** Open this notebook from its **Open in Colab** badge above
> and the data loads itself from GitHub.

> No prior Python needed — every block of code is explained in the comments (the grey text
> after a `#`). Python ignores those; they are notes for humans.

## Step 1 — Set up our tools

In [ ]:
# ---- Load the toolboxes we need ---------------------------------------------
# 'import X as Y' loads library X and gives it the short nickname Y.
#   pandas       : spreadsheets/tables in Python. One table is called a "DataFrame".
#   numpy        : fast maths on numbers. Here we mainly meet np.nan, which means "blank".
#   matplotlib   : draws charts.
# All three are pre-installed on Google Colab, so there is nothing to install.
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ---- scikit-learn: the standard machine-learning library ---------------------
# It is huge, so we import only the three small pieces this lab uses.
from sklearn.linear_model import LinearRegression      # fits a straight-line price model
from sklearn.model_selection import train_test_split   # splits data into "train" and "test"
from sklearn.metrics import mean_squared_error         # measures how wrong the model is

# ---- One display setting -----------------------------------------------------
# Show up to 20 columns when printing a table instead of hiding some behind "...".
pd.set_option("display.max_columns", 20)

print("Libraries loaded. Ready to go.")

## Step 2 — Load the dirty data

The data lives in the course GitHub repository, and the cell below reads it **straight from
the internet**. There is no file to download, no folder to pick and no upload button — and it
behaves identically on **Mac, Windows and Linux**, because no file path is ever involved.

In [ ]:
# ---- Where the data comes from ----------------------------------------------
# DATA_URL is the file's RAW address on GitHub. The difference matters:
#   raw.githubusercontent.com/...  -> the actual file contents  (what pandas needs)
#   github.com/...                 -> the web PAGE about the file (pandas cannot read it)
# If you ever swap in your own file, copy the link from GitHub's "Raw" button.
DATA_URL  = "https://raw.githubusercontent.com/cyberirishman/5-day-AI-Cyber/main/homes_dirty.csv"
DATA_FILE = "homes_dirty.csv"    # only used by the offline fallbacks below

import os   # 'os' lets us ask the computer whether a file exists on disk

# ---- A loader that tries the internet first, then falls back ------------------
# You do not need to understand this function to do the lab. It exists so the
# notebook still works if the room's wifi is down or you are running offline.
def load_csv(url="", fname=""):
    """Load the CSV from a URL, or a local file, or (on Colab) an upload box."""
    # 1st choice: read straight from the web address.
    if url:
        try:
            return pd.read_csv(url)
        except Exception as problem:
            print("Could not read from the internet:", problem)
            print("Falling back to a local copy...")
    # 2nd choice: a copy sitting next to this notebook, or inside a data/ folder.
    for path in [fname, os.path.join("data", fname)]:
        if fname and os.path.exists(path):
            return pd.read_csv(path)
    # 3rd choice (Colab only): pop up an upload button and read whatever is chosen.
    from google.colab import files
    uploaded = files.upload()
    return pd.read_csv(list(uploaded.keys())[0])

# ---- Actually load it --------------------------------------------------------
# pd.read_csv() turns a comma-separated file into a DataFrame (a table).
dirty = load_csv(DATA_URL, DATA_FILE)

print("Loaded", len(dirty), "rows.")   # len() of a table = its number of rows
dirty.head()                           # .head() shows the first 5 rows so we can eyeball it

**What you should see:** `Loaded 1015 rows.` and a small table with four columns —
`bedrooms`, `sqft`, `age_years`, `price`. Look at the `price` column: some values have a
`$` and a comma in them. That is our first clue.

## Step 3 — What type is each column?

Before asking anything of the data, ask what the data *is*. Every column has a **type**, and
pandas guesses it when the file is read. One bad value in a column is enough to change that
guess for the whole column — which is exactly what has happened to `price` here.

In [ ]:
# ---- What type is each column? -----------------------------------------------
# .dtypes lists the data type pandas chose for every column.
#   int64 / float64 = "whole number" / "decimal number"  -> we CAN do maths on these
#   object / str    = TEXT                               -> we CANNOT
# Watch the 'price' row. It only takes a handful of text values in a column of a
# thousand numbers for pandas to give up and call the whole column text.
print(dirty.dtypes, "\n")

# ---- Who is the culprit? ------------------------------------------------------
# .str.contains() searches inside each value and answers True/False.
# r"[\$,]" is a pattern meaning "contains a $ or a , character".
looks_like_text = dirty["price"].astype(str).str.contains(r"[\$,]", regex=True)

print("Prices stored as text:", looks_like_text.sum(), "out of", len(dirty))

# ---- Where are they, so you can go and look? -----------------------------------
# pandas numbers its rows from 0 and does not count the header line of the file.
# So a row's LINE NUMBER in homes_dirty.csv is its pandas index + 2:
#     +1 because pandas starts at 0 and a file starts at 1
#     +1 for the header line "bedrooms,sqft,age_years,price"
# That is the same number VS Code shows in its left margin and the same row number
# a spreadsheet shows. We never renumber the rows in this lab, so these line numbers
# stay correct all the way to the end -- handy for checking our work later.
print("Examples -- open homes_dirty.csv at these line numbers:")
for i in dirty.index[looks_like_text][:3]:
    print("   line {:<5}  {}".format(i + 2, dirty.loc[i, "price"]))

**What you should see:** `price` reported as `object` (or `str` on the newest pandas) — both
mean **text** — and **40** prices written the way a human writes them, like `$158,437`.

That is the whole problem in one line: **you cannot take the average of a piece of text.**

## Step 3b — Ask the anchor question anyway

We want a number on screen, so we reach for the blunt instrument:
`pd.to_numeric(..., errors="coerce")`. It tries to convert every value, and `coerce` says
*"whatever I can't convert, leave a blank"*.

> **This is input validation, not cleaning — and the distinction matters.**
> Refusing to feed text into a maths function is good defensive coding, the same reflex as
> validating input before it reaches a database query. It protects the *operation*.
> But it does nothing for the *data*: the values it refuses are silently discarded, and
> `.mean()` then averages whatever survived without mentioning the loss. Input validation
> that drops records without telling anyone is how evidence quietly disappears — worth
> remembering when we get to log integrity and poisoning on **Day 3**.

In [ ]:
# ---- A crude way to get numbers: to_numeric with errors="coerce" --------------
# to_numeric() tries to convert EVERY value in the column. It is not hunting for
# dollar signs -- it has no idea what a dollar sign is. It simply tries, and
# errors="coerce" says: whatever I cannot convert, leave a blank (NaN) instead.
# "$300,000" fails that attempt, so it is NOT repaired here -- it is thrown away.
# THIS IS NOT CLEANING. It is a guard rail: it stops text from reaching .mean() and
# breaking it -- good practice, the same instinct as validating input before it hits
# a database. It protects the calculation, not the data. The real fix -- deleting the
# $ and the , so the value survives as a number -- is Step 4.
price_num = pd.to_numeric(dirty["price"], errors="coerce")

# ---- Count what the guard rail just threw away ---------------------------------
# dirty["bedrooms"] == 4 makes a column of True/False, one per row ("is this a 4-bed?").
four_bed = dirty["bedrooms"] == 4
lost = price_num[four_bed].isna().sum()      # .isna() = "is this a blank?", .sum() counts them
print("4-bedroom houses in the file :", four_bed.sum())
print("...with no usable price      :", lost, "(discarded, not fixed)")

# ---- The anchor question, on dirty data ----------------------------------------
# price_num[four_bed] keeps only the prices where the answer was True.
# .mean() then averages what is left -- quietly skipping every blank.
dirty_4bed_mean = price_num[four_bed].mean()

# "{:,.0f}" formats the number with thousands separators and no decimal places.
print("\nDIRTY answer -> mean price of a 4-bedroom house: ${:,.0f}".format(dirty_4bed_mean))

That number is **absurd** — no normal 4-bedroom house averages near a million dollars.

And notice the answer is biased in **both** directions at once:

- the prices we threw away were the *ordinary-looking* ones (`$158,437`, `$178,499`) — a
  human typed them nicely, so the computer refused them;
- the `$88,000,000` typo converts perfectly well, so it **stayed**.

We removed good data and kept the bad. This is *garbage in, garbage out*. Now let's actually
clean, one step at a time.

## Step 4 — Fix wrong types / units

Strip the `$` and commas out of `price`, then convert every column to real numbers.
`.str.replace(r"[\$,]", "", regex=True)` deletes any `$` or `,` characters.

In [ ]:
# ---- Work on a copy -----------------------------------------------------------
# .copy() gives us a separate table, so 'dirty' stays untouched and we can always
# compare before-and-after. Without .copy() we would be editing the original.
clean = dirty.copy()

# ---- Strip the $ and , out of price, then make it a real number ----------------
# .astype(str)  : treat every value as text so the next line always works.
# .str.replace(): find-and-replace inside text. r"[\$,]" is a pattern meaning
#                 "a $ character or a , character"; we replace them with nothing.
clean["price"] = clean["price"].astype(str).str.replace(r"[\$,]", "", regex=True)
clean["price"] = pd.to_numeric(clean["price"], errors="coerce")

# ---- Convert the other three columns to numbers as well ------------------------
# A 'for' loop repeats the same instruction for each name in the list.
# Anything that still is not a number (blank, junk text) becomes NaN = blank.
for col in ["bedrooms", "sqft", "age_years"]:
    clean[col] = pd.to_numeric(clean[col], errors="coerce")

print("price is now:", clean["price"].dtype)

# ---- Re-ask the anchor question ------------------------------------------------
# .loc[rows, column] selects rows that match a condition AND one named column.
print("After fixing types -> 4-bed mean: ${:,.0f}".format(
      clean.loc[clean["bedrooms"] == 4, "price"].mean()))

# ---- What did this step NOT fix? -----------------------------------------------
# .nlargest(3) returns the three biggest prices, and .items() hands us each row's
# index with its value -- so we can print the line number (index + 2) alongside.
# These are typing mistakes, not expensive houses, and converting them to numbers
# did nothing to help: they are perfectly valid numbers. They go in Step 7.
print("\nThree biggest prices still in the data:")
for i, v in clean["price"].nlargest(3).items():
    print("   line {:<5} ${:>12,.0f}".format(i + 2, v))

**What you should see:** `price is now: float64` — a real number at last. The 4-bed mean has
moved, because the `"$..."` prices we were silently dropping are now counted.

It is still far too high, and the printout shows exactly why: the three **"extra-zeros" typos**
— **line 470 ($88,000,000)**, **line 729 ($62,000,000)** and **line 313 ($50,000,000)** — are
still sitting in the data. They converted perfectly well, because they *are* valid numbers.
They are just wrong. Step 7 is where they go.

## Step 5 — Handle missing values

A blank (`NaN`) price or size can't help us. Count them, then drop rows missing any core field.
`.isna().sum()` counts blanks per column; `.dropna(subset=[...])` removes rows with a blank in
those columns.

In [ ]:
# ---- Count the blanks ----------------------------------------------------------
# .isna() turns every cell into True (blank) or False (has a value).
# .sum() then counts the Trues, column by column -- True counts as 1, False as 0.
print("Missing values per column:")
print(clean[["bedrooms", "sqft", "age_years", "price"]].isna().sum(), "\n")

# ---- Drop rows that are blank in any core field --------------------------------
# subset=[...] means "only look at these columns when deciding what to drop".
# A house with no price and no size cannot teach a model anything.
clean = clean.dropna(subset=["bedrooms", "sqft", "age_years", "price"])

print("After dropping missing -> 4-bed mean: ${:,.0f}".format(
      clean.loc[clean["bedrooms"] == 4, "price"].mean()))

## Step 6 — Drop impossible rows

Some values are physically impossible: negative age, `40` bedrooms, a `99999` sqft placeholder.
We keep only rows inside sane ranges. The `&` means "and"; each condition is wrapped in `()`.

In [ ]:
# ---- Remember how many rows we had, so we can report how many we drop ----------
before = len(clean)

# ---- Keep only rows where every condition is True -------------------------------
# Each line makes a True/False column; & combines them with "and".
# The brackets around each condition are NOT optional in pandas -- without them
# Python applies & in the wrong order and you get a confusing error.
clean = clean[
    (clean["age_years"] >= 0) &          # age cannot be negative
    (clean["bedrooms"].between(1, 10)) & # .between(a, b) means a <= value <= b. 40 beds is not a house.
    (clean["sqft"].between(200, 10000))  # 200-10000 sqft is a real house; 99999 is a placeholder
]

print("Dropped", before - len(clean), "impossible rows.")
print("After dropping impossible -> 4-bed mean: ${:,.0f}".format(
      clean.loc[clean["bedrooms"] == 4, "price"].mean()))

## Step 7 — Remove outliers with the IQR rule

The three extra-zeros typos are still here. A standard, no-guesswork way to flag outliers is the
**IQR (Inter-Quartile Range)** rule: find the 25th percentile (Q1) and 75th percentile (Q3);
anything more than `1.5 × (Q3 − Q1)` below Q1 or above Q3 is an outlier. It's the same maths
behind the "whiskers" on a box-plot.

> Don't be alarmed when the printed *lower* bound comes out **negative**. It just means no
> house in this dataset is cheap enough to count as a low-side outlier — the rule is only
> catching the absurdly expensive ones.

In [ ]:
# ---- Work out where "normal" ends ----------------------------------------------
# .quantile(0.25) returns the price that 25% of houses are cheaper than.
q1 = clean["price"].quantile(0.25)     # 25% of houses cost less than this
q3 = clean["price"].quantile(0.75)     # 75% of houses cost less than this
iqr = q3 - q1                          # the spread of the "middle 50%" of houses

# The 1.5 is the standard convention -- the same one that draws box-plot whiskers.
low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
print("Keeping prices between ${:,.0f} and ${:,.0f}".format(low, high))

# ---- Name the rows the rule is about to remove ----------------------------------
# ~ means "not", so ~between(low, high) picks the rows OUTSIDE the band.
# Printing index + 2 gives the line number in homes_dirty.csv again, so you can open
# the file and confirm for yourself that the rule removed the right rows -- and only
# those. A cleaning step you cannot audit is a cleaning step you should not trust.
outliers = clean.loc[~clean["price"].between(low, high), "price"]
print("Dropping", len(outliers), "price outliers:")
for i, v in outliers.items():
    print("   line {:<5} ${:>12,.0f}".format(i + 2, v))

# ---- Keep only prices inside that band -----------------------------------------
before = len(clean)
clean = clean[clean["price"].between(low, high)]

print("After removing outliers -> 4-bed mean: ${:,.0f}".format(
      clean.loc[clean["bedrooms"] == 4, "price"].mean()))

**What you should see:** this is the step where the answer *snaps*. A handful of typo rows were
holding the average hostage — notice how few rows we actually dropped to achieve it.

## Step 8 — Remove duplicate rows

The same house imported twice quietly double-counts. `.drop_duplicates()` keeps one copy of
each identical row.

In [ ]:
# ---- Remove exact repeats ------------------------------------------------------
# .drop_duplicates() compares whole rows and keeps only the first copy of each.
# Duplicates do not usually move an average much, but they make a model over-confident
# about the houses it has seen twice.
before = len(clean)
clean = clean.drop_duplicates()

print("Dropped", before - len(clean), "duplicate rows.")
print("FINAL cleaned answer -> 4-bed mean: ${:,.0f}".format(
      clean.loc[clean["bedrooms"] == 4, "price"].mean()))

### The anchor answer converged

| Stage | Mean price of a 4-bedroom house |
|---|---|
| Dirty data | **absurd (~$897,000)** |
| after fixing types | ~$863,000 — still absurd |
| after missing / impossible rows | ~$894,000 — barely moves |
| after removing outliers | **snaps to ~$190,000** |
| after de-duplicating | **~$190,000 (stable)** |

Notice *where* the number moved. Three of the four cleaning steps barely touched it — the
whole distortion came from **three** outlier rows out of a thousand.

Same question, same houses — a believable answer only *after* cleaning. Cleaning didn't tidy
the data; it **changed the answer**.

## Step 9 — Does cleaning change a *model* too?

Averages are simple. Let's check a real model. We fit a **linear regression** — a straight-line
formula that predicts `price` from `bedrooms`, `sqft`, and `age_years` — once on the dirty data
and once on the clean data, and compare the error (**RMSE**, roughly the typical dollars-off per
prediction; lower is better).

In [ ]:
# ---- A small reusable function -------------------------------------------------
# 'def' defines a function: a named block of code we can run more than once.
def fit_and_score(frame, label):
    """Train a linear model to predict price, return its test-set RMSE (typical $ error)."""
    X = frame[["bedrooms", "sqft", "age_years"]]   # inputs (the "features")
    y = frame["price"]                             # the thing we predict (the "label")

    # Hold back 20% of the houses that the model never sees while training, so we can
    # grade it on fresh examples. random_state=42 just fixes which houses are held back,
    # so everyone in the room gets the identical split and the identical number.
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

    # .fit() IS the training step: it finds the straight line that best matches the data.
    model = LinearRegression().fit(X_tr, y_tr)

    # Grade it: compare predictions on the held-back houses to their real prices.
    # ** 0.5 means "square root" -- that is what turns MSE into RMSE, back in dollars.
    rmse = mean_squared_error(y_te, model.predict(X_te)) ** 0.5
    print("{:<8} RMSE: ${:,.0f}".format(label, rmse))
    return model

# ---- Build a *naively* numeric version of the DIRTY data -------------------------
# This is what you get if you skip real cleaning and just force everything to numbers:
# text prices coerced, blanks filled with the median, outliers and duplicates left in.
dirty_naive = dirty.copy()
for col in ["price", "bedrooms", "sqft", "age_years"]:
    dirty_naive[col] = pd.to_numeric(
        dirty_naive[col].astype(str).str.replace(r"[\$,]", "", regex=True), errors="coerce")
dirty_naive = dirty_naive.fillna(dirty_naive.median(numeric_only=True))

# ---- Same model, same features, two different datasets ---------------------------
_ = fit_and_score(dirty_naive, "DIRTY")     # the _ means "we don't need this model later"
clean_model = fit_and_score(clean, "CLEAN")  # we DO keep this one, to draw its line in Step 10

**What you should see:** roughly **$6,100,000** typical error on the dirty data against about
**$20,000** on the clean data — the clean model's error is a *tiny fraction* of the dirty one's.
Outliers and blanks don't just skew an average; they wreck what a model learns.

## Step 10 — See it: dirty vs. clean

In [ ]:
# ---- One figure, two panels side by side -----------------------------------------
# plt.subplots(1, 2) makes 1 row and 2 columns of charts; ax[0] is left, ax[1] is right.
fig, ax = plt.subplots(1, 2, figsize=(11, 4))

# ---- LEFT panel: the dirty data ----------------------------------------------------
# .scatter() draws one dot per house: sqft across, price up. The huge typo prices push
# the y-axis so high that every real house is squashed into a line along the bottom.
ax[0].scatter(dirty_naive["sqft"], dirty_naive["price"], s=8, alpha=0.5, color="#C0392B")
ax[0].set_title("DIRTY: a few typos dominate the whole chart")
ax[0].set_xlabel("sqft"); ax[0].set_ylabel("price ($)")

# ---- RIGHT panel: the clean data, plus the line our model learned -------------------
ax[1].scatter(clean["sqft"], clean["price"], s=8, alpha=0.5, color="#1E5199")

# np.linspace(a, b, 100) makes 100 evenly spaced sqft values from smallest to largest.
xs = np.linspace(clean["sqft"].min(), clean["sqft"].max(), 100)

# To draw the line we must give the model all three inputs, so we vary sqft while
# holding bedrooms and age at their average values.
grid = pd.DataFrame({"bedrooms": clean["bedrooms"].mean(), "sqft": xs,
                     "age_years": clean["age_years"].mean()})
ax[1].plot(xs, clean_model.predict(grid), color="#00838F", lw=2, label="fitted line")
ax[1].set_title("CLEAN: a sensible price pattern"); ax[1].set_xlabel("sqft"); ax[1].legend()

plt.tight_layout()   # stop the two titles overlapping
plt.show()           # display the figure

## Wrap-up

- Cleaning is **not** cosmetic — it moved the mean 4-bed price from ~$897k to ~$190k, and cut
  the model's typical error from ~$6.1M to ~$20k.
- The four dirt types (wrong type, missing, outlier, duplicate) each did real damage — but the
  **outliers** did nearly all of it here, and there were only three of them.
- **Security tie-in:** malformed or *poisoned* logs corrupt a detector the exact same way —
  we come back to deliberate data poisoning on **Day 3**.
- **Next (Lab 3b):** the data is clean, but the columns are on wildly different scales
  (bedrooms 1-6 vs. sqft in the thousands). We'll see why that alone can blind a model — and
  fix it with **normalization**.